In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ProToxin)

This notebook curates the **ProToxin** dataset by parsing multiple FASTA files, inferring binary labels from file names, running duplicate consistency checks, generating metadata, and exporting standardized datasets for downstream modeling.

 - **Toxic effect / endpoint:** toxic
- **Source:** ProToxin
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Reads ProToxin FASTA files** from the source directory.
  - All FASTA files except `bt_all.fasta` are treated as **labeled** inputs.
  - `bt_all.fasta` is treated as an **unlabeled** pool and exported separately.
- **Infers labels from file names** for labeled files:
  - files containing `"neg"` → `label = 0`
  - otherwise → `label = 1`
- **Creates an unlabeled dataset**:
  - sequences from `bt_all.fasta` are assigned `label = 2` to explicitly mark unknown class membership.
- **Checks duplicated sequences** for both labeled and unlabeled sets:
  - duplicates with consistent labels are collapsed,
  - conflicting duplicates are flagged and exported as errors.
- **Builds metadata** using the project-wide Excel description sheet and appends QC counters.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv` (binary labeled set)
  - `detected_unlabel_sequences.csv` (label = 2)
  - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "ProToxin"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
dfs = []
for file in (Path(PATH_INPUT) / name_source).glob("*"):
    if file.name != "bt_all.fasta":  # Excluide the file
        df = read_fasta_doc(file)
        df["source_file"] = file.name
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [4]:
df = (
    df
    .assign(
        label=lambda d: d["source_file"]
            .str.contains("neg", case=False, na=False)
            .map({True: 0, False: 1})
    )
    [["sequence","label"]]
)
df.shape

(264597, 2)

In [5]:
df_unlabel = (
    read_fasta_doc(f"{PATH_INPUT}/{name_source}/bt_all.fasta")
    .assign(label=2) # There is no information about the labels of this source, therefore it will be identified with a 2
    [["sequence", "label"]]
)

- Checking duplicates

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_remove_duplicated_unlabel, df_errors_unlabel, df_unique_unlabel = processing_duplicated(df_unlabel, group_seq="sequence", sort_key="label")
df_full_unlabel = pd.concat([df_unique_unlabel, df_remove_duplicated_unlabel], axis=0)

In [8]:
df_errors.shape

(1737, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
raw_total_sequences = (len(df) + len(df_unlabel))

In [11]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": int(len(df_full) + len(df_full_unlabel)),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "number_of_unlabel_sequences" : len(df_full_unlabel),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 6, 25, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'No information;Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from uniprot',
 'repository or server': 'https://www.yanglab-mi.org.cn/ProToxin/',
 'publication': 'https://www.mdpi.com/2072-6651/17/10/489',
 'number_of_raw_sequences': 265350,
 'number_of_sequences_retained': 252536,
 'number_of_positive_sequences': 6954,
 'number_of_negative_sequences': 244829,
 'number_of_erroneous_sequences': 1737,
 'number_of_unlabel_sequences': 753,
 'modified_sequences_included': False}

- Exporting data

In [12]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [13]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_full_unlabel.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_unlabel_sequences.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)